# Bronze Table Inspection

Ad-hoc scan of `raw_data/bronze/runs`, the Delta table written by the `bronze_runs` Dagster asset (`sts_pipeline/assets/bronze.py`). This is a dev/inspection notebook, not part of the numbered analysis sequence — it exists to eyeball the raw ingested data, not to produce analysis output.

In [1]:
import os
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up.
PROJECT_ROOT = Path.cwd().parent

os.environ["JAVA_HOME"] = str(PROJECT_ROOT / ".jdk17" / "jdk-17.0.20+8")
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = str(PROJECT_ROOT / ".venv" / "Scripts") + os.pathsep + r"C:\hadoop\bin" + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")

BRONZE_RUNS_PATH = str(PROJECT_ROOT / "raw_data" / "bronze" / "runs")

In [2]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder.master("local[*]")
    .appName("bronze-inspection")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "8g")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(BRONZE_RUNS_PATH)
print(f"Loaded {BRONZE_RUNS_PATH}")

Loaded e:\Projects\sts-card-choice-analysis\raw_data\bronze\runs


## Schema

Nested columns (`card_choices`, `relics_obtained`, `campfire_choices`, etc.) stay as arrays/structs at bronze rather than being flattened — that reshaping happens at silver.

In [3]:
df.printSchema()

root
 |-- ascension_level: long (nullable = true)
 |-- boss_relics: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- not_picked: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- picked: string (nullable = true)
 |-- build_version: string (nullable = true)
 |-- campfire_choices: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- data: string (nullable = true)
 |    |    |-- floor: double (nullable = true)
 |    |    |-- key: string (nullable = true)
 |-- campfire_rested: long (nullable = true)
 |-- campfire_upgraded: long (nullable = true)
 |-- card_choices: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- floor: double (nullable = true)
 |    |    |-- not_picked: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- picked: string (nullable = true)
 |-- character_chosen: string (nullabl

## Row count and quick sanity checks

Comparable to the checks in `01_data_collection.ipynb` (win rate, floor distribution) — just confirming the Spark ingestion landed the same shape of data as the old pandas pass did.

In [ ]:
print("Row count:", df.count())
print()
print("victory value counts:")
df.groupBy("victory").count().show()
print("character_chosen value counts:")
df.groupBy("character_chosen").count().orderBy("count", ascending=False).show()

In [4]:
df.filter(F.col("play_id") == "f5b09f88-5853-4bc0-9e7c-f28223ee4b0a") \
    .select(F.explode("card_choices").alias("c")) \
    .select("c.floor", "c.picked", "c.not_picked", F.size("c.not_picked").alias("n_not_picked")) \
    .show(truncate=False)

+-----+---------------+-----------------------------------+------------+
|floor|picked         |not_picked                         |n_not_picked|
+-----+---------------+-----------------------------------+------------+
|1.0  |SKIP           |[Rip and Tear, Leap, Sweeping Beam]|3           |
|6.0  |Amplify        |[Lockon, Chaos]                    |2           |
|7.0  |Redo           |[Barrage, Rebound]                 |2           |
|10.0 |Glacier        |[Cold Snap, Turbo]                 |2           |
|11.0 |Stack          |[Conserve Battery, Doom and Gloom] |2           |
|12.0 |Steam          |[Creative AI, Ball Lightning]      |2           |
|13.0 |SKIP           |[Leap, Turbo, Steam Power]         |3           |
|14.0 |Hello World    |[Conserve Battery, Compile Driver] |2           |
|15.0 |Go for the Eyes|[Conserve Battery, Coolheaded]     |2           |
|16.0 |Fission        |[Core Surge, All For One]          |2           |
|18.0 |Sweeping Beam+1|[Melter, Recycle]           

## Table view

`.toPandas()` on a small, flat-column slice renders as a normal table in Jupyter. Nested columns are left out here since they'd just show as raw list/struct reprs — see the next section for those.

In [ ]:
flat_cols = [
    "play_id", "character_chosen", "ascension_level", "victory",
    "floor_reached", "score", "build_version", "_source_file",
]
df.select(flat_cols).limit(50).toPandas()

## Peek at a nested column

`card_choices` is the field the whole analysis hinges on — confirm it looks like the `{picked, not_picked, floor}` shape from `01_data_collection.ipynb`.

In [ ]:
sample_row = df.select("play_id", "card_choices").filter(df.card_choices.isNotNull()).first()
print("play_id:", sample_row["play_id"])
for pick in sample_row["card_choices"]:
    print(pick)

## Stop Spark

Run this when done exploring — otherwise the JVM stays alive holding memory until the kernel is restarted.

In [ ]:
spark.stop()